# Sequence Prediction - N-BACK protocol, WITHOUT app metadata

Copy of `_results_sequence_prediction_nback2.ipynb` with the concatenated
app-metadata (SBERT) half removed: the projector + LSTM are trained *and*
evaluated on the Qwen screen embeddings alone.

Every stage still runs with `--nback K`, which predicts a screen from **exactly
`K` preceding screens**, sliding over the trace, so the target is always the
*end of the chain*. `--nback` is an alternative to `--strict`, not a variant of
it; the two flags are mutually exclusive.

| | permissive (default notebook) | `--strict` | `--nback K` (this notebook) |
|---|---|---|---|
| traces kept | >= 3 screens | >= `n_predictors + 1` (= 5) | >= `K + 1` |
| context per prediction | 1..4 screens (truncated near trace start) | exactly 4 | exactly `K` |
| target | next screen | next screen, full windows only | end of the `K`-chain, full windows only |
| candidate pool | every embedding in the directory | screens of eligible traces | screens of eligible traces |
| top-k% thresholds | `ceil()` | `int()` truncation (`modeltester.py:141`) | `ceil()` |
| training negatives | any screen | screens of eligible traces | screens of eligible traces |

### What is removed, and why

`6_generate_app_context_embeddings.py` embeds each screen's Play-Store app
description with SBERT (`sentence-transformers/all-mpnet-base-v2`, 768-d) and
concatenates that vector onto the projected screen vector. It is a *per-app*
constant: every screen of the same app carries the identical second half.

| stage | sibling notebook (with app desc) | this notebook (`_wo_app`) |
|---|---|---|
| `9_train_joint.py` | 4096 -> `PROJ_DIM` projector + LSTM | **identical** - training never sees the app half |
| `6_generate_app_context_embeddings.py` | concatenates the 768-d SBERT app vector onto every screen vector | **not run** |
| `4_evaluate_screen_prediction.py` | retrieval over `PROJ_DIM + 768`-d vectors | retrieval over `PROJ_DIM`-d screen vectors |

The LSTM itself is unaffected by the removal, because it never consumed the app
half in the first place: `9_train_joint.py` trains on the raw Qwen embeddings
only, and at evaluation time `4_evaluate_screen_prediction.py` slices the screen
part off (`ctx_raw[:, :lstm_dim]`) before the forward pass. The app half enters
*only* in the ranking step, where it is copied verbatim from the **target** onto
the prediction (`4_evaluate_screen_prediction.py:373-375`). That is what these
notebooks remove: the retrieval query no longer carries the target's own app
identity.

Why it matters (see `SUMMARY.md`): with the app half attached, every candidate
from the target's app scores high, and since a RICO app has ~7.1 screens on
average (median 5), the Top-0.01% ... Top-5% columns largely measure app
identity rather than screen prediction - a measured app-only oracle already
reaches 0.9968 at Top-0.1% and 1.0 at Top-1%. In this notebook **every** column,
not just Top-1, is a screen-level number.

No code change is needed for any of this: when the embedding dimension equals
`lstm_dim`, the concatenation branch in the evaluator simply does not fire.

**Artefacts.** `9_train_joint.py` writes exactly the same files as the sibling
notebook, so the training cell below reuses them when they are already on disk.
Nothing is written under a `_with_app_desc` name and nothing here can clobber a
with-app run - the whole difference is which directory the evaluator is pointed
at.

    embeddings_seq_768_nback2/                 (9_, shared with the nback2 sibling)
    lstm_predictor_768d_nback2.pth             (9_, shared with the nback2 sibling)
    sequence_projector_768d_nback2.pth         (9_, shared with the nback2 sibling)
    embeddings_seq_768_nback2_with_app_desc/   (6_)  - NOT used here

### Caveats on the numbers this produces

1. **Screen ordering.** Screen2Vec builds each trace from an unsorted
   `os.listdir()` of `view_hierarchies`
   (`UI_embedding/precompute_embeddings.py:72`), which is not temporal order and
   is not reproducible on another filesystem. This pipeline always uses
   `gestures.json` order, i.e. true temporal order.

2. **Not comparable to the paper's own Top-k%.** Screen2Vec's Table 3 numbers
   were produced *with* the concatenated app-description half. Compare this
   notebook against its `_with_app_desc` sibling to read the size of that effect;
   do not put a `_wo_app` Top-1% next to a paper Top-1% and call it a
   reproduction.

3. **Trace and layout id namespaces are joined through the index.** Embeddings
   are keyed by Rico's *global* `data/layouts/` numbering while `gestures.json`
   keys are per-trace indices; every stage here resolves them through
   `trace_layout_index.csv` (`0_build_trace_index.py`) and refuses to run without
   it. Unmatched trace screens are dropped, so the evaluated population is the
   matched subset.

4. **Numbers from different `K` are not directly comparable.** The eligible-trace
   population shrinks as `K` grows (>= `K + 1` screens), and the candidate pool -
   and therefore the top-k% thresholds and the retrieval difficulty - shrinks
   with it: ~64.4k pool screens at `K=1`, ~61.7k at `K=2`, ~54.5k at `K=4`. A
   change in accuracy across `K` mixes the effect of context length with the
   effect of an easier pool. Read each `K` on its own.


In [1]:
# ==== Configuration: set N here, everything below follows from it ====
NBACK = 2        # number of preceding screens used as context (>= 1)
PROJ_DIM = 768   # projector output dimension

EMB_DIR = f"embeddings_seq_{PROJ_DIM}_nback{NBACK}"          # 9_ output, evaluated as-is
LSTM_PATH = f"lstm_predictor_{PROJ_DIM}d_nback{NBACK}.pth"   # 9_ output
LABEL = f">>> Qwen3-VL + LSTM (Ours, no app desc) [nback={NBACK}] <<<"

print(f"n-back = {NBACK}  |  proj_dim = {PROJ_DIM}  |  app metadata = OFF")
print(f"  projected embeddings : {EMB_DIR}/")
print(f"  LSTM checkpoint      : {LSTM_PATH}")


n-back = 2  |  proj_dim = 768  |  app metadata = OFF
  projected embeddings : embeddings_seq_768_nback2/
  LSTM checkpoint      : lstm_predictor_768d_nback2.pth


In [2]:
# 9_train_joint.py is IDENTICAL to the with-app sibling notebook: it trains on the
# raw Qwen embeddings and never sees the app half, so it writes the very same
# artefacts. Reuse them when they are already on disk; delete one to force a retrain.
import os

if os.path.isfile(LSTM_PATH) and os.path.isdir(EMB_DIR):
    print(f"Reusing existing artefacts:\n  {LSTM_PATH}\n  {EMB_DIR}/")
    print("(delete either one to retrain from scratch)")
else:
    get_ipython().run_line_magic("run", f"9_train_joint.py --proj_dim {PROJ_DIM} --nback {NBACK}")


Reusing existing artefacts:
  lstm_predictor_768d_nback2.pth
  embeddings_seq_768_nback2/
(delete either one to retrain from scratch)


In [3]:
# 6_generate_app_context_embeddings.py is deliberately NOT run here. That stage is
# the only place the SBERT app-description vector enters the pipeline, so skipping
# it - and pointing the evaluator at EMB_DIR instead of EMB_DIR_with_app_desc - is
# the whole of the `_wo_app` variant.


In [ ]:
%run 4_evaluate_screen_prediction.py --embeddings_dir $EMB_DIR --lstm_model $LSTM_PATH --nback $NBACK --label "$LABEL"



[Step 0/3] Loading LSTM predictor from 'lstm_predictor_768d_nback2.pth'...
  Loaded LSTM predictor (embed_dim=768, best_val_loss=2.2094)

[Step 1/3] Loading embeddings...
Loading 66261 embeddings from 'embeddings_seq_768_nback2' in parallel...


Loading embeddings: 100%|██████████| 66261/66261 [00:55<00:00, 1190.63it/s]
